In [2]:
from IPython.display import Image

- 每当有新的算法或者新的硬件-感知(infra)的算法，都是重新理解 Attention/residual 或者 Transformer 架构的视角和机会
    - mla，mhc，attention residual
    - flash attention，infra 层面的优化

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V
$$

$$
\text{Attention}( \underbrace{Q}_{N \times d_k} , \underbrace{K}_{M \times d_k} , \underbrace{V}_{M \times d_v} ) = \text{softmax} \underbrace{\left( \frac{\overbrace{Q}^{N \times d_k} \overbrace{K^T}^{d_k \times M}}{\sqrt{d_k}} \right)}_{N \times M} \underbrace{V}_{M \times d_v} = \underbrace{O}_{N \times d_v}
$$

- K/V 是配对的（长度是一致的，矩阵乘法的角度，序列长度要对齐），kv cache
- 在标准的自注意力机制中，如果一次性输入整段文本进行处理，则有 $N=M$，而在交叉注意力（Cross-Attention）或带有 KV Cache 的生成阶段中(decoding, 1对多)，N 和 M 可以不相等。

- prefill (ttft, time to first token) vs. decode (generation speed)
    - inference pipeline
        - prefill -> (sample -> decode) -> (sample -> decode)....
        - kv cache，空间换时间
            - 预填充，填充的对象即是，history/context tokens 的 kv cache
    - 并行 / 串行
        - gemm / gemv (General Matrix-Matrix Multiplication, General Matrix-Vector Multiplication)
            - gemm: `[L, D]` -> layer weights (matrix) -> `[L, D]` -> ...
            - gemv: `[1, D]` -> layer weights (matrix) -> `[1, D]` -> ...
            - 权重复用的视角
        - seq modeling: position encoding, causal mask
            - prefill 更接近于训练（训练时的 forward），decode 是存推理
            - Transformer 的高度并行化，体现在其 training 和 inference 的 prefill 阶段
    - Arithmetic Intensity: $\text{AI} = \frac{\text{FLOPs}}{\text{Bytes moved}}$
        - memory wall/bound，computation bound
        - gpu memory, gpu utilization
    - infra
        - flash attention：prefill，online softmax 算子融合
        - vllm (inference engine)：
            - paged attention: decode 节省显存
            - continuous batching: (1, d) => (B, d)
            - chunked prefill: Prefill & Decode 混合调度
        - ...

In [1]:
from IPython.display import Image

In [3]:
Image(url='./figs/prefill-decode.png', width=800)

| 特征 | Prefill (Context) | Decode (Generation) |
| :--- | :--- | :--- |
| **数学运算类型** | **GEMM** (矩阵-矩阵乘法) | **GEMV** (矩阵-向量乘法) |
| **输入形状** | $B \times L \times D$ (所有 Prompt Token) | $B \times 1 \times D$ (仅当前 Token) |
| **Attention 维度** | $L \times L$ (一次算全图，prompt seq) | $1 \times (L + t)$ (Query 查历史 Cache) |
| **Causal 实现** | 显式 Mask 矩阵 (Masked Fill) | 隐式 (只拿当前 Query 查过去的 KV) |
| **瓶颈 (Bottleneck)** | **Compute-bound** (算力) | **Memory-bound** (带宽) |
| **GPU 利用率** | 高 (Volta/Ampere Tensor Cores 跑满) | 低 (卡在显存读取上) |
| **KV Cache** | 写入 (Write Only) | 读取 + 追加 (Read + Append) |
| **nvitop: gpu-util** | **瞬间飙升 / 接近 100%**<br>因为是计算密集型，Tensor Core 全速运转。 | **相对较低 / 波动 (Sawtooth)**<br>因为是访存密集型，计算核心常处于空闲等待数据状态。 |
| **nvitop: memory** | **阶梯式突增 (Step Jump)**<br>瞬间申请 Prompt 所有 Token 的 KV Cache。 | **缓慢线性增长 (Creeping)**<br>随生成长度 $t$ 增加，逐个 Token 累积 KV Cache。 |

- Prefill 阶段 (计算密集)
    - 当一个新的请求（特别是长 Prompt）进来时，你会看到 GPU 利用率瞬间打满（绿色条拉满），同时显存占用会突然“跳”上去一截。
    - 此时 GPU 正在并行计算整个 Prompt 的 Attention，这是一个巨大的矩阵乘法操作，非常适合 GPU 的并行架构。显存的跳变是因为系统一次性为 Prompt 中的所有 Token 分配并写入了 KV Cache。
- Decode 阶段 (访存密集)
    - 紧接着 Prefill 之后，GPU 利用率通常会下降，表现为中等负载或锯齿状波动（取决于 Batch Size 和调度策略）。同时，显存占用会像爬楼梯一样，一点一点缓慢增加。
    - 此时每次只生成一个 Token。虽然计算量很小，但 GPU 必须从显存中把之前所有历史的 KV Cache 搬运到计算单元。“搬运数据的时间”远大于“计算的时间”，导致 GPU 核心在空转（Memory Wall），所以利用率上不去。
- 计算 / 内存 => Arithmetic Intensity
    - prefill：吃算力，要算得快
    - decode：吃带宽，要搬得快
    - 苹果芯片的迭代

### Prefill 阶段：并行计算 (Compute-bound)

推理的第一步，目的是“阅读”并理解用户输入的 Prompt。
- 输入形式：假设输入 Prompt 长度为 $L$，Batch Size 为 $B$，Hidden Dimension 为 $D$。输入矩阵 $X \in \mathbb{R}^{B \times L \times D}$。
- 计算特点：GEMM (General Matrix-Matrix Multiplication)。
    - 我们可以一次性计算所有 $L$ 个 Token 的 Query, Key, Value 向量。
    - $Q = XW_Q, K = XW_K, V = XW_V$，其中 $W \in \mathbb{R}^{D \times D}$。
        - position encoding
    - 此时，$Q, K, V$ 都是 $\mathbb{R}^{B \times L \times D}$ 的矩阵。

Causal Attention 与 Mask (掩码)
- 在 Prefill 阶段，虽然所有 Token 是并行进入计算的，但模型必须遵守“因果律”（Causal）：第 $t$ 个 Token 只能看到它之前的 $0...t$ 个 Token，不能看到 $t+1$ 之后的。
- 这是通过 Causal Mask 实现的。 注意力分数计算：
    - $A = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)$
    - 其中 $M$ 是一个 $L \times L$ 的下三角掩码矩阵：
        - $M_{ij} =
\begin{cases}
0 & \text{if } i \geq j \text{ (Visible)}\\
-\infty & \text{if } i < j \text{ (Masked)}
\end{cases}$

由于 $M$ 的存在，Attention 矩阵 $A$ 的上三角部分被 Mask 掉（概率为 0）。
- 并行性：极高。计算 $QK^T$ 是一个 $B \times H \times L \times L$ 的矩阵乘法。GPU 可以火力全开，算力利用率（Utilization）很高。
- 瓶颈：通常是 Compute-bound（计算受限）。

### Decode 阶段：串行自回归 (Memory-bound)

生成 Response 的过程，逐个 Token 生成。
- 输入形式：当前时刻 $t$（$t > L$），输入仅仅是上一个生成的 Token $x_t$。输入向量 $x_t \in \mathbb{R}^{B \times 1 \times D}$。
- 计算特点：GEMV (General Matrix-Vector Multiplication)。
    - 我们只需要计算当前这 1个 Token 的 $q_t, k_t, v_t$。$q_t = x_t W_Q$。
    - 此时 $q_t$ 只是一个向量（或者 $B \times 1 \times D$ 的扁平矩阵）。

KV Cache 与 Attention
- 在 Decode 阶段，我们不需要重新计算之前 $0...t-1$ 时刻的 Key 和 Value，而是直接从显存中读取（这就是 KV Cache）。
- 当前步计算：
    - 计算当前 Token 的 $q_t, k_t, v_t$。
    - 将 $k_t, v_t$ 拼接到 KV Cache 中：
        - $K_{\text{cache}} \leftarrow [K_{\text{old}}; k_t], \quad V_{\text{cache}} \leftarrow [V_{\text{old}}; v_t]$
        - 此时 $K_{\text{cache}}$ 的形状是 $B \times (L+t) \times D$。
    - Attention 计算：
        - $a_t = \text{Softmax}\left(\frac{q_t K_{\text{cache}}^T}{\sqrt{d_k}}\right) \cdot V_{\text{cache}}$
        - 注意这里是 向量 $q_t$ 去乘 矩阵 $K_{\text{cache}}^T$。
- 并行性：时间维度无法并行。必须等 $t$ 生成完，才能生成 $t+1$。但在 Batch 维度和 Head 维度依然可以并行。
- 瓶颈：Memory-bound（显存带宽受限）。
    - 为了计算仅仅 1 个 Token 的输出，你需要把庞大的 KV Cache（可能几十 GB）从 HBM (High Bandwidth Memory) 搬运到 GPU 核心中进行一次计算，算完就扔回去了。计算量小，搬运量大，导致 arithmetic intensity（算术强度）极低。

### 权重复用

> 片外显存（HBM）与片上高速缓存（SRAM）之间的数据搬运鸿沟（显存带宽）。

- GPU 的计算单元（Tensor Cores/CUDA Cores）速度极快，但它的“操作台”——片上缓存（SRAM）容量极小（通常只有几十 MB）。而大模型的权重矩阵 $W$（动辄几十甚至上百 GB）只能存放在外部的“大仓库”——显存（HBM）中。计算单元无法直接读取 HBM 中的数据。每一次计算，必须先将 $W$ 的一小块（Block）像吊车搬运巨型模具一样，从 HBM 缓慢搬运到 SRAM 中，才能进行运算。
- Prefill 阶段：高效的“批发”模式（权重复用）
    - 在 Prefill 阶段，输入是一个长度为 $L$ 的序列矩阵 $X$（维度为 $L \times d$）。当底层算子（如 cuBLAS）执行 $X \cdot W$ 的矩阵乘法时，它会采用分块（Tiling）策略：从 HBM 中搬运一小块权重 $W_{block}$ 到 SRAM 中（这消耗了大量时间和带宽）。权重复用的时刻： 这块好不容易搬来的 $W_{block}$，会立刻与输入矩阵 $X$ 中的整整 $L$ 个 Token（即 $X$ 的 $L$ 个不同的行向量）分别进行乘加运算。也就是说，这块“模具”被搬上操作台后，连续“冲压”了 $L$ 个不同的零件，才被移走换下一块。
    - 数学视角的复用率：每搬运 $1$ Byte 的权重，执行了 $L$ 次计算。搬运成本被 $L$ 个 Token 完美“平摊”了。这就是所谓的权重复用极高。此时系统是 Compute-Bound（计算单元在疯狂运转，搬运带宽绰绰有余）。
- Decode 阶段：奢侈的“零售”模式（权重无法复用）
    - 在 Decode 阶段，输入退化成了当前生成的唯一一个 Token 的向量 $X$（维度为 $1 \times d$）。此时的计算变成了矩阵乘向量（GEMV）：同样从 HBM 中搬运那一小块权重 $W_{block}$ 到 SRAM 中。复用的失效： 这块 $W_{block}$ 只能与当前的这 $1$ 个 Token 进行一次点积运算。算完这唯一的一次后，这块权重在 SRAM 中的使命就结束了，必须立刻被丢弃，以便为下一块权重腾出空间。
    - 数学视角的复用率：每搬运 $1$ Byte 的权重，仅仅执行了 $1$ 次计算。搬运巨型“模具”的巨大成本，全部由这 $1$ 个可怜的 Token 承担。这就是所谓的权重无法复用。此时系统陷入 Memory-Bound（计算单元瞬间算完，然后绝大部分时间都在干等下一块权重从 HBM 慢吞吞地搬过来）。

### GEMM vs. GEMV

> Prefill: General Matrix Multiply, Decode: General Matrix-Vector Multiply

- Prefill 阶段的任务：计算完所有输入 Token 后，将它们的 Key 和 Value 向量保存起来供后续使用。
    - 在正式开始一个一个吐字（Decode）之前，先预先计算并将必要的上下文信息填充到显存中。
    - 如果没有这个阶段，模型在生成每一个新 Token 时，都必须把之前所有的输入提示词重新计算一遍，这会造成巨大的算力浪费和不可接受的延迟。
- prefill & decode
    - 在 Prefill 阶段，模型接收了长度为 $L$ 的 Prompt 序列。经过深层 Transformer 网络的流水线计算，模型不仅在显存中存下了这 $L$ 个 Token 的动态 KV Cache，还在最后一层输出了一个形状为 $(L, d)$ 的隐藏状态矩阵（Hidden States）。
    - 为了预测“下一个词”，系统通常只取用这个矩阵的最后一行（即第 $L$ 个 Token 对应的 $1 \times d$ 向量），将其送入语言模型头（LM Head）。LM Head 本质上是一个形状为 $(d, V)$ 的线性分类层，其中 $V$ 是模型支持的词表大小（Vocabulary Size）。
    - 经过 LM Head 的矩阵乘法和 Softmax 激活函数后，Prefill 阶段的最终输出是一个形状为 $(1, V)$ 的概率分布向量。这个向量代表了词汇表中每一个候选词成为“序列中第一个新词”的概率。
    - 得到 $(1, V)$ 的概率分布后，系统必须进行一次“离散化”的抉择：
        - 采样（Sampling / Argmax）：根据系统设定的 Temperature、Top-K 或 Top-P 策略，从这个概率分布中挑选出一个唯一的词。此时，连续的概率分布坍缩成了一个离散的整数索引——Token ID（例如，数字 4096 可能代表单词 "The"）。
        - 词嵌入查找（Embedding Lookup）：拿到这个整数 Token ID 后，系统会去模型的静态词嵌入表（Embedding Matrix，其形状同样为 $(V, d)$）中，直接“抽出”第 4096 行。


### Arithmetic Intensity

$$
\text{AI} = \frac{\text{FLOPs}}{\text{Bytes moved}}
$$

算力强度（Arithmetic Intensity, AI），即“每从显存（hbm）读取 1 Byte 数据，能执行（tensor core）多少次浮点运算（FLOPs）”。以 FP16 精度（每个参数 2 Bytes）为例：
- Prefill
    - $\text{FLOPs} = (L \cdot d_{out}) \cdot (2 \cdot d) = 2 \cdot L \cdot d \cdot d_{out}$
        - 在计算 $Y = X \cdot W$ 时，输出矩阵 $Y$ 中的每一个元素 $y_{ij}$，都是由输入矩阵 $X$ 的第 $i$ 行与权重矩阵 $W$ 的第 $j$ 列进行**点积（Dot Product）**得到的。
        - 这两个向量的长度都是 $d$。计算一次长度为 $d$ 的点积，需要进行 $d$ 次乘法和 $d-1$ 次加法。
        - 在计算机底层架构的性能评估中，乘加累积操作（MAC, Multiply-Accumulate）通常被视为 2 次浮点运算（1 次乘 + 1 次加）。因此，计算 $Y$ 中的一个元素，需要近似 $2 \cdot d$ 次 FLOPs。
    - 内存读取量：$2 \cdot (L \cdot d + d \cdot d_{out} + L \cdot d_{out})$
        - 完整的矩阵乘法需要完成三次物流搬运：读取输入 $X$：总共有 $L \cdot d$ 个元素，搬运量为 $2 \cdot L \cdot d$ Bytes。
        - 读取权重 $W$：总共有 $d \cdot d_{out}$ 个元素，搬运量为 $2 \cdot d \cdot d_{out}$ Bytes。
        - 写回输出 $Y$：计算完成后，需要将结果存回显存，总共有 $L \cdot d_{out}$ 个元素，搬运量为 $2 \cdot L \cdot d_{out}$ Bytes。
    - 算力强度：当输入序列 $L$ 较大时，权重 $W$ 的读取时间被 $L$ 个 Token 平摊。
    $$\text{AI} = \frac{2 \cdot L \cdot d \cdot d_{out}}{2 \cdot (L \cdot d + d \cdot d_{out} + L \cdot d_{out})} = \frac{L \cdot d \cdot d_{out}}{L \cdot d + d \cdot d_{out} + L \cdot d_{out}}= \frac{L}{1 + L \cdot (\frac{1}{d} + \frac{1}{d_{out}})}
    $$
    - $\lim_{L \to \infty} \text{AI} = \frac{1}{\frac{1}{d} + \frac{1}{d_{out}}} = \frac{d \cdot d_{out}}{d + d_{out}}=\frac d2$
- decode
    - 在 Decode 阶段，由于每次只处理当前生成的一个 Token，输入不再是一个长序列矩阵，而退化成了一个行向量。
        - $X$ (输入向量)：维度为 $(1, d)$。代表当前这唯一一个 Token 的特征表达。
        - $W$ (权重矩阵)：维度依然为 $(d, d_{out})$。模型参数是固定的，体积极大。
        - $Y$ (输出向量)：维度为 $(1, d_{out})$。
    - $\text{FLOPs} = 1 \cdot d_{out} \cdot (2 \cdot d) = 2 \cdot d \cdot d_{out}$
    - 内存读取量：$2 \cdot (d + d \cdot d_{out} + d_{out}) \approx 2 \cdot d \cdot d_{out}$ （必须把庞大的权重矩阵完整搬运到计算单元）
        - 计算这一个 Token，我们需要完成三次 HBM（显存）的数据搬运：
        - 读取输入 $X$：只有 1 个 Token，搬运量为 $2 \cdot (1 \cdot d) = 2 \cdot d$ Bytes。
        - 读取权重 $W$：这是最致命的一步。为了计算这仅仅一个 Token 的结果，必须把整个庞大的权重矩阵从头到尾完整读取一遍。搬运量为 $2 \cdot d \cdot d_{out}$ Bytes。
        - 写回输出 $Y$：生成的结果写入显存，搬运量为 $2 \cdot (1 \cdot d_{out}) = 2 \cdot d_{out}$ Bytes。
    - 算力强度：$\text{AI} \approx \frac{2 \cdot d \cdot d_{out}}{2 \cdot d \cdot d_{out}} = 1$ FLOP/Byte。
        - 意味着 GPU 的计算单元每从显存里千辛万苦搬运 1 Byte 的数据过来，只能执行 1 次微不足道的浮点计算。
    - 仅仅看线性层是不够的。大模型的灵魂在于自注意力（Self-Attention），而在 Decode 阶段，自注意力的计算是整个系统最棘手的瓶颈。当前 Token 的 Query 向量 $Q$ 维度为 $(1, d)$。历史 KV Cache 中，$K$ 和 $V$ 的矩阵维度均为 $(L, d)$。
        - 总 $\text{FLOPs} = (2 \cdot d \cdot L + 2 \cdot L \cdot d) = 4 \cdot L \cdot d$
        - 读取 KV Cache 的总字节数 = 每个用户 $L$ 个 Token $\times$ $2$ (Key和Value) $\times$ $d$ 维 $\times$ $2$ Bytes/FP16 = $4 \cdot L \cdot d$ Bytes。

### Roof-line Model

In [7]:
Image(url='./figs/roofline.png', width=500)

> 注意 FLOPs 表示复数的概念，多少次，FLOPS 是速度的概念，每秒多少次，也就是所谓的算力，TFLOPS）；

$$
I = \frac{\text{FLOPs}}{\text{Bytes}}
$$

- 算力密度（通常记为 $I$）描述了算法本身的计算密集程度，它是一个纯粹的软件/算法指标，与硬件无关。其数学定义为：算法在运行过程中，每从内存（DRAM）中读取/写入 1 Byte 的数据，能够执行的浮点运算次数（FLOPs）。
- Roofline 模型将硬件的物理极限（算力和带宽）与算法的算力密度结合在了一个二维坐标系中，用以预测在特定硬件上算法能达到的真实性能上限（$P$）。

$$
P = \min(P_{\text{peak}}, I \cdot BW_{\text{peak}})
$$

- $P_{\text{peak}}$：硬件的峰值算力（例如 TFLOPS），即“房顶”的平坦部分，代表计算墙（Compute Wall）。
    - 常量
- $BW_{\text{peak}}$（bandwidth）：硬件的峰值内存带宽（例如 GB/s），结合算力密度 $I$ 构成了倾斜的上升线，代表内存墙（Memory Wall）。
- 拐点（Ridge Point）：当 $I = \frac{P_{\text{peak}}}{BW_{\text{peak}}}$ 时，算法恰好同时榨干了硬件的计算力和带宽。
    - 当算法的 $I$ 位于拐点左侧：算法处于**内存受限（Memory-bound）**状态。此时，即便增加再多的计算核心（如 GPU 的 CUDA 核心），性能也不会提升，因为数据喂不进去。
    - 当算法的 $I$ 位于拐点右侧：算法处于**计算受限（Compute-bound）**状态。此时，数据供应充足，瓶颈在于处理器的计算速度。

### 系统优化

- FlashAttention：在 Prefill 阶段收益巨大，因为它把 $O(L^2)$ 的显存读写变成了分块计算，解决了 Prefill 的显存爆炸问题。
    - online softmax 使得算子可以融合
- PagedAttention (vLLM)：主要优化 Decode 阶段。因为 Decode 瓶颈在显存带宽和显存碎片化，PagedAttention 通过非连续内存管理，让 KV Cache 读取更高效，从而支持更大的 Batch Size 来掩盖 Decode 的低效。
- Chunked Prefill：为了防止 Prefill 占用太久导致 Decode 任务卡顿（Head-of-line blocking），将 Prefill 拆成小块和 Decode 穿插执行。
    - Prefill 是 Compute-bound 的高密度矩阵乘法 $\mathbf{Q}\mathbf{K}^T$；而 Decode 是 Memory-bound。当系统正在流畅处理几十个并发的 Decode 时，如果突然塞入一个携带 32K token 的超长新请求，传统调度器必须让 GPU 停下所有 Decode 任务，去全力“咀嚼”这个巨大的 Prefill。这会导致正在输出的几十个请求突然卡顿，Inter-Token Latency (ITL) 剧烈飙升。
    - 调度器会在同一个 Forward Pass 中，精妙地将切分后的 Prefill 块与 Decode Tokens 拼接组装：

$$
\text{Batch}_{mixed} = \begin{bmatrix} \mathbf{X}_{prefill\_chunk} \\ \mathbf{X}_{decode} \end{bmatrix} \in \mathbb{R}^{(C+B) \times d}
$$
- 这不仅平滑了延迟（极大地降低了 ITL），还实现了一种奇妙的“化学反应”：将 Compute-bound 的 Prefill 计算与 Memory-bound 的 Decode 访存完美融合，让 GPU 的 Tensor Cores 和显存总线在同一周期内同时达到满载饱和状态。